In [2]:
file_path = '../bathroomCabinet_10/hand_landmarks.txt'

count = 0
with open(file_path, 'r') as file:
    for line in file:
        if line.strip().startswith("landmark {"):
            count += 1

print(f"Number of landmarks: {count}")


Number of landmarks: 64071


In [3]:
import cv2

video_path = "../bathroomCabinet_10.mp4"
cap = cv2.VideoCapture(video_path)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total number of frames: {frame_count}")

cap.release()


Total number of frames: 5476


In [2]:
import numpy as np

def calculate_hand_acceleration_magnitude(landmarks, fps=30):
    """
    Calculate acceleration magnitude for 3D hand landmark positions.
    
    Args:
        landmarks: List of dictionaries with 'x', 'y', 'z' keys for each frame
        fps: Frame rate (default 30 fps for EgoPAT-3D)
    
    Returns:
        acceleration_magnitudes: List of signed acceleration magnitudes
                               (+ve = acceleration, -ve = deceleration/retardation)
    """
    if len(landmarks) < 3:
        raise ValueError("Need at least 3 frames to calculate acceleration")
    
    # Convert to numpy array for vectorized operations
    positions = np.array([[lm['x'], lm['y'], lm['z']] for lm in landmarks])
    
    # Calculate time step
    dt = 1.0 / fps
    
    # First derivative: velocity = dp/dt
    velocities = np.diff(positions, axis=0) / dt
    
    # Second derivative: acceleration = dv/dt
    accelerations = np.diff(velocities, axis=0) / dt
    
    # Calculate acceleration magnitudes
    accel_magnitudes = np.linalg.norm(accelerations, axis=1)
    
    # Determine sign: positive for acceleration, negative for retardation
    # Compare speed changes between consecutive frames
    speed_prev = np.linalg.norm(velocities[:-1], axis=1)
    speed_curr = np.linalg.norm(velocities[1:], axis=1)
    speed_change = speed_curr - speed_prev
    
    # Apply sign based on speed change
    signed_acceleration = accel_magnitudes * np.sign(speed_change)
    
    return signed_acceleration.tolist()

def get_acceleration_for_frame_filtering(landmarks, fps=30, threshold=None):
    """
    Enhanced function for frame filtering in zero-shot TIL research.
    
    Args:
        landmarks: List of landmark dictionaries
        fps: Frame rate 
        threshold: Optional threshold for filtering high-motion frames
    
    Returns:
        dict: Contains accelerations, frame indices, and filtering info
    """
    accelerations = calculate_hand_acceleration_magnitude(landmarks, fps)
    
    # Frame indices (acceleration starts from frame 2)
    frame_indices = list(range(2, len(landmarks)))
    
    result = {
        'accelerations': accelerations,
        'frame_indices': frame_indices,
        'abs_accelerations': [abs(a) for a in accelerations]
    }
    
    if threshold is not None:
        # Identify high-motion frames for filtering
        high_motion_frames = [
            idx for idx, acc in zip(frame_indices, accelerations) 
            if abs(acc) > threshold
        ]
        result['high_motion_frames'] = high_motion_frames
        result['motion_percentage'] = len(high_motion_frames) / len(accelerations) * 100
    
    return result

# Example usage for your research:
example_landmarks = [
    {'x': 0.280, 'y': 0.640, 'z': -0.020},  # Start
    {'x': 0.281, 'y': 0.641, 'z': -0.021},  # Slow start
    {'x': 0.290, 'y': 0.650, 'z': -0.030},  # Sudden acceleration
    {'x': 0.305, 'y': 0.665, 'z': -0.045},  # Peak speed
    {'x': 0.310, 'y': 0.668, 'z': -0.048},  # Start braking
    {'x': 0.312, 'y': 0.669, 'z': -0.049},  # Sharp deceleration
    {'x': 0.3125, 'y': 0.6695, 'z': -0.0495}  # Nearly stopped
]
# Calculate accelerations
accelerations = calculate_hand_acceleration_magnitude(example_landmarks)
print("Acceleration magnitudes:", accelerations)

# For frame filtering (useful for your zero-shot TIL research)
filtering_results = get_acceleration_for_frame_filtering(example_landmarks, threshold=0.5)
print("Filtering results:", filtering_results)


Acceleration magnitudes: [12.470765814495893, 9.353074360871974, -17.727944043233, -3.710795063055898, -1.4924811556599462]
Filtering results: {'accelerations': [12.470765814495893, 9.353074360871974, -17.727944043233, -3.710795063055898, -1.4924811556599462], 'frame_indices': [2, 3, 4, 5, 6], 'abs_accelerations': [12.470765814495893, 9.353074360871974, 17.727944043233, 3.710795063055898, 1.4924811556599462], 'high_motion_frames': [2, 3, 4, 5, 6], 'motion_percentage': 100.0}


In [3]:
import cv2

# Input and output paths
input_path = '../bathroomCabinet_10.mp4'
output_path = 'first10clips.mp4'

# Open the input video
cap = cv2.VideoCapture(input_path)

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4

# Output writer
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Frame counter
frame_count = 0
max_frame = 536  # inclusive; will write frames 0 to 536

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_count > max_frame:
        break
    out.write(frame)
    frame_count += 1

# Release resources
cap.release()
out.release()
print(f"Video trimmed and saved to: {output_path}")


Video trimmed and saved to: first10clips.mp4


In [4]:
import cv2

# Define frame ranges (inclusive)
frame_ranges = [
    (79, 120),
    (127, 164),
    (172, 215),
    (223, 262),
    (272, 309),
    (321, 353),
    (363, 402),
    (408, 450),
    (462, 501),
    (510, 536)
]

# Input and output video paths
input_path = '../test_data/first10clips.mp4'
output_path = '../test_data/trimmed_first10clips.mp4'

# Open input video
cap = cv2.VideoCapture(input_path)
if not cap.isOpened():
    raise IOError("Error opening video file")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4

# Create output video writer
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Process each range
for start, end in frame_ranges:
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    for frame_num in range(start, end + 1):  # Inclusive
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)

# Release everything
cap.release()
out.release()
print(f"Trimmed video saved to: {output_path}")


Trimmed video saved to: ../test_data/trimmed_first10clips.mp4


In [ ]:
import cv2

# Define frame ranges (inclusive)
frame_ranges = [
    (79, 120),
    (127, 164),
    (172, 215),
    (223, 262),
    (272, 309),
    (321, 353),
    (363, 402),
    (408, 450),
    (462, 501),
    (510, 536)
]

# Input and output video paths
input_path = '../test_data/first10clips.mp4'
output_path = '../test_data/trimmed_first10clips.mp4'

# Open input video
cap = cv2.VideoCapture(input_path)
if not cap.isOpened():
    raise IOError("Error opening video file")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4

# Create output video writer
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Font settings
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1.0
font_color = (255, 255, 255)  # white
thickness = 2

# Padding from top-right corner
margin_x = 10
margin_y = 30

# Process each range
for start, end in frame_ranges:
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    for frame_num in range(start, end + 1):  # Inclusive
        ret, frame = cap.read()
        if not ret:
            break

        # Frame number as text
        text = f"Frame: {frame_num}"

        # Get text size to align right
        text_size, _ = cv2.getTextSize(text, font, font_scale, thickness)
        text_x = width - text_size[0] - margin_x
        text_y = margin_y

        # Overlay text
        cv2.putText(frame, text, (text_x, text_y), font, font_scale, font_color, thickness)

        # Write frame to output
        out.write(frame)

# Release everything
cap.release()
out.release()
print(f"Video saved with frame numbers at: {output_path}")


Video saved with frame numbers at: ../test_data/trimmed_first10clips.mp4


In [6]:
def extract_5th_landmark(input_file, output_file):
    with open(input_file, 'r') as f:
        lines = f.readlines()

    output_lines = []
    current_frame = None
    landmark_counter = 0
    capture = False
    landmark_data = []

    for line in lines:
        line = line.strip()

        # Detect frame start
        if line.endswith(':'):
            current_frame = line
            landmark_counter = 0
            capture = False
            continue

        # Detect landmark start
        if line == 'landmark {':
            landmark_counter += 1
            capture = (landmark_counter == 5)
            landmark_data = ['landmark {']
            continue

        if capture:
            landmark_data.append(line)

            if line == '}':
                # Save only the 5th landmark of the current frame
                output_lines.append(current_frame)
                output_lines.extend(landmark_data)
                output_lines.append("")  # Blank line between frames
                capture = False

    # Write output to new file
    with open(output_file, 'w') as f:
        f.write('\n'.join(output_lines))

    print(f"Extracted 5th landmark from each frame to: {output_file}")


# Example usage
extract_5th_landmark('../bathroomCabinet_10/hand_landmarks.txt', '../test_data/5th_landmarks.txt')


Extracted 5th landmark from each frame to: ../test_data/5th_landmarks.txt


In [10]:
import cv2
import re

def get_required_frame_numbers(txt_path):
    """Extract frame numbers from the text file"""
    frame_numbers = []
    with open(txt_path, 'r') as f:
        for line in f:
            match = re.match(r'^(\d+):', line.strip())
            if match:
                frame_numbers.append(int(match.group(1)))
    return frame_numbers

def trim_video_by_frames(video_path, output_path, frame_numbers):
    """Trim and save only specified frames"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError("Could not open video file")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for frame_num in sorted(frame_numbers):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if not ret:
            print(f"Warning: Could not read frame {frame_num}")
            continue

        # Optional: Add frame number on top-right corner
        text = f"Frame: {frame_num}"
        text_size, _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        text_x = width - text_size[0] - 10
        text_y = 30
        cv2.putText(frame, text, (text_x, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"Trimmed video saved to: {output_path}")


# Usage
landmark_txt = '../test_data/5th_landmarks_trimmed.txt'
input_video = '../test_data/first10clips.mp4'
output_video = '../test_data/landmarked_first10clips.mp4'

frames_to_keep = get_required_frame_numbers(landmark_txt)
trim_video_by_frames(input_video, output_video, frames_to_keep)


Trimmed video saved to: ../test_data/landmarked_first10clips.mp4
